# logit-thermo A2-7b：本滴定の分析（CPU）

- **必要な secret：なし**。**ランタイム：CPU（標準）**。見込みは 10〜20 分。**データ集めの 3 本が全部終わってから**回してください。
- `A2_7B/` の 3 モデル × 8 本の腕を読み、判定を計算して `A2_7B/a2_7b_summary.json` に書きます。**結果は表示しません。**
- 計算のコード：A2-2・A2-4 の分析のコードは Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_7b_analyze.py`（`d4da9ac740e2b645…`）は埋め込み（データを開く前に固定）。
- **通知**：🚀・🎉・⚠️。名札は「logit-thermo A2-7b 本滴定 分析（判定と要約、CPU）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-7b 本滴定 分析（判定と要約、CPU）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-7b analyze', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：分析のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRCA22': 'fe1f96db1f7e84031e85a70bcc8dcbb95c9fc06b3367657ff4074a9f9d1343f7', 'SRCA24': 'e70bad0db9b7780c2059aae52ad06debe87757bc582791deefb58c5f435f84a6'})
SRCA22, SRCA24 = _s['SRCA22'], _s['SRCA24']
SRC7BA = '"""A2-7b（ゲート docs/A2_7B_GATE.md）の分析：本滴定。データを開く前に固定する。CPU。\nscripts/a2_2_analyze.py（pav・crossing_mono・feedback_stats・per_doc_feedback・NBOOT）、scripts/a2_4_analyze.py（stats_fixed・paired・\nverdict_inf・copy_frac）を実行したのと同じ名前空間で実行する。\n"""\nimport math\nimport numpy as np\n\nB7A_MODELS = [\'pythia-1b\', \'olmo2-1b\', \'qwen2.5-1.5b\']\nB7A_TEMPS = [0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 1.0]\nB7A_ARMS = [\'none\', \'none_b\', \'cq33\', \'cq67\', \'cq100\', \'pl025\', \'pl050\', \'pl075\']\nB7A_COPY, B7A_PL = [\'cq33\', \'cq67\', \'cq100\'], [\'pl025\', \'pl050\', \'pl075\']\nB7A_MASS_T = [0.8, 0.85, 0.9]\nB7A_BOOT, B7A_BOOT_SEED = 1000, 20261111\nB7A_Q = 1.5\nDOWN_WORDS = (\'大きく下がる\', \'大きく下がる（格子の外）\', \'少し下がる\')\n\n\ndef load_7b(d, m):\n    Zt = np.load(f\'{d}/{m}@tf.npz\')\n    assert [float(t) for t in Zt[\'temps\']] == B7A_TEMPS\n    tf = Zt[\'tf\']                                                   # (nD, 8, L)\n    use = [int(u) for u in Zt[\'use\']]\n    fds, mass_doc, extra = {}, {}, {}\n    kk = [B7A_TEMPS.index(t) for t in B7A_MASS_T]\n    for a in B7A_ARMS:\n        Z = np.load(f\'{d}/{m}@{a}.npz\')\n        assert [float(t) for t in Z[\'temps\']] == B7A_TEMPS and [int(u) for u in Z[\'use\']] == use, (m, a)\n        fds[a] = per_doc_feedback(tf, Z[\'g_ell\'])\n        bm = Z[\'g_ban_mass\'].astype(np.float64)                     # (8, S, nD, L)\n        mass_doc[a] = bm[kk].mean(axis=(0, 1, 3))                   # (nD,)：記事ごとの量（温度 0.8・0.85・0.9、サンプル、歩で平均）\n        extra[a] = dict(toks=Z[\'toks\'], prompts=Z[\'prompts\'], top1=Z[\'g_banned_top1\'].astype(np.float64).mean(axis=(1, 2, 3)).tolist())\n    return fds, mass_doc, extra\n\n\ndef slope0(x, y):\n    """原点を通る直線 y ＝ −K x の K（最小二乗）。"""\n    x, y = np.asarray(x, float), np.asarray(y, float)\n    return float(-(x @ y) / (x @ x)) if (x @ x) > 0 else float(\'nan\')\n\n\ndef log_q(Kc, Kp):\n    if not (np.isfinite(Kc) and np.isfinite(Kp)):\n        return float(\'nan\')\n    if Kc > 0 and Kp > 0:\n        return math.log(Kc / Kp)\n    if Kc > 0 >= Kp:\n        return float(\'inf\')\n    if Kp > 0 >= Kc:\n        return float(\'-inf\')\n    return float(\'nan\')\n\n\ndef fit_once(fds, mass_doc, eps, idx, arms_c, arms_p):\n    """idx の記事で、腕ごとの T_0 と量 → (K_写し, K_偽薬, 使った腕で交点が全部出たか)。"""\n    T0n = crossing_mono(np.median(fds[\'none\'][idx], 0), B7A_TEMPS, eps)\n    if not np.isfinite(T0n):\n        return float(\'nan\'), float(\'nan\'), False\n    out, ok = {}, True\n    for a in arms_c + arms_p:\n        T0 = crossing_mono(np.median(fds[a][idx], 0), B7A_TEMPS, eps)\n        if not np.isfinite(T0):\n            ok = False\n            continue\n        out[a] = (float(mass_doc[a][idx].mean()), math.log(T0 / T0n))\n    Kc = slope0([out[a][0] for a in arms_c if a in out], [out[a][1] for a in arms_c if a in out])\n    Kp = slope0([out[a][0] for a in arms_p if a in out], [out[a][1] for a in arms_p if a in out])\n    return Kc, Kp, ok\n\n\ndef bq1_word(lq_point, ci, fail_frac, enough):\n    if not enough or fail_frac > 0.1 or not (np.isfinite(ci[0]) or np.isinf(ci[0])):\n        return \'はっきりしない\'\n    lo, hi = ci\n    if lo > 0 and lq_point >= math.log(B7A_Q) - 1e-12:\n        return \'写しの禁止のほうがよく効く\'\n    if lo >= -math.log(B7A_Q) and hi <= math.log(B7A_Q):\n        return \'同じくらい（束一的）\'\n    if hi < 0 and lq_point <= -math.log(B7A_Q) + 1e-12:\n        return \'偽薬のほうがよく効く\'\n    return \'はっきりしない\'\n\n\ndef analyze7b(d):\n    res = {\'models\': {}}\n    for m in B7A_MODELS:\n        fds, mass_doc, extra = load_7b(d, m)\n        eps = feedback_stats(fds[\'none\'], B7A_TEMPS, np.random.default_rng(0))[\'eps\']\n        s0 = stats_fixed(fds[\'none\'], B7A_TEMPS, eps)\n        row = {\'eps\': eps, \'arms\': {}}\n        k07 = B7A_TEMPS.index(0.7)\n        for a in B7A_ARMS:\n            s = stats_fixed(fds[a], B7A_TEMPS, eps)\n            toks, prm = extra[a][\'toks\'][k07], extra[a][\'prompts\']\n            dd = dict(T0=s[\'T0\'], cls=s[\'cls\'], F=s[\'F\'], mass=float(mass_doc[a].mean()), top1_rate_by_T=extra[a][\'top1\'],\n                      y=(math.log(s[\'T0\'] / s0[\'T0\']) if np.isfinite(s[\'T0\']) and np.isfinite(s0[\'T0\']) else float(\'nan\')),\n                      copy4_07=float(np.mean([copy_frac(prm[j], toks[q, j], 4) for q in range(toks.shape[0]) for j in range(toks.shape[1])])),\n                      copy3_07=float(np.mean([copy_frac(prm[j], toks[q, j], 3) for q in range(toks.shape[0]) for j in range(toks.shape[1])])))\n            if a != \'none\':\n                p = paired(fds[a], fds[\'none\'], B7A_TEMPS, eps, seed=1)\n                dd.update(d_vs_none=p, word=verdict_inf(p, s[\'cls\'], np.isfinite(s0[\'T0\'])))\n            row[\'arms\'][a] = dd\n        # BQ1：点の当てはめに使う腕 ＝ 交点が出た腕\n        arms_c = [a for a in B7A_COPY if np.isfinite(row[\'arms\'][a][\'T0\'])]\n        arms_p = [a for a in B7A_PL if np.isfinite(row[\'arms\'][a][\'T0\'])]\n        enough = len(arms_c) >= 2 and len(arms_p) >= 2 and np.isfinite(s0[\'T0\'])\n        nD = fds[\'none\'].shape[0]\n        Kc, Kp, _ = fit_once(fds, mass_doc, eps, np.arange(nD), arms_c, arms_p) if enough else (float(\'nan\'),) * 3\n        lq = log_q(Kc, Kp)\n        rng = np.random.default_rng(B7A_BOOT_SEED)\n        lqs, fail = [], 0\n        for _ in range(B7A_BOOT):\n            idx = rng.integers(0, nD, nD)\n            if not enough:\n                break\n            kc, kp, ok = fit_once(fds, mass_doc, eps, idx, arms_c, arms_p)\n            if not ok:\n                fail += 1\n                continue\n            v = log_q(kc, kp)\n            if not np.isnan(v):\n                lqs.append(v)\n        lqs = np.array(lqs)\n        ci = ([float(np.percentile(lqs, 2.5, method=\'lower\')), float(np.percentile(lqs, 97.5, method=\'higher\'))]\n              if len(lqs) >= 20 else [float(\'nan\'), float(\'nan\')])\n        ff = fail / B7A_BOOT if enough else 1.0\n        row[\'BQ1\'] = dict(arms_copy=arms_c, arms_placebo=arms_p, K_copy=Kc, K_placebo=Kp, logQ=lq, Q=(math.exp(lq) if np.isfinite(lq) else lq),\n                          ci_logQ=ci, boot_fail=ff, n_boot_used=int(len(lqs)), word=bq1_word(lq, ci, ff, enough))\n        row[\'BQ2_word\'] = row[\'arms\'][\'pl075\'][\'word\']\n        res[\'models\'][m] = row\n    res[\'BQ1\'] = sum(res[\'models\'][m][\'BQ1\'][\'word\'] == \'写しの禁止のほうがよく効く\' for m in B7A_MODELS)\n    res[\'BQ2\'] = sum(res[\'models\'][m][\'BQ2_word\'] in DOWN_WORDS for m in B7A_MODELS)\n    return res\n\n\ndef self_test_a27b_analyze():\n    """作り物：記事ごとの f（8 温度）を、T_0 ＝ T0n × exp(y) で作る。写しは y ＝ −5 × 量、偽薬は y ＝ −1.5 × 量 → Q ≈ 3.3 で「写し」。\n    偽薬も −5 × 量 → 「同じくらい」。"""\n    import tempfile, os\n    t = np.array(B7A_TEMPS)\n    rng = np.random.default_rng(0)\n    nD, S, L = 40, 2, 24\n\n    def write(dirn, m, Kc, Kp, T0n=0.86):\n        tf = np.zeros((nD, 8, L), np.float32)\n        np.savez(f\'{dirn}/{m}@tf.npz\', tf=tf, temps=np.array(B7A_TEMPS), use=np.arange(nD))\n        masses = dict(none=0.0, none_b=0.0, cq33=0.014, cq67=0.028, cq100=0.042, pl025=0.014, pl050=0.029, pl075=0.043)\n        for a, mass in masses.items():\n            K = Kc if a.startswith(\'cq\') else Kp if a.startswith(\'pl\') else 0.0\n            T0 = T0n * math.exp(-K * mass)\n            F = np.where(t < T0, (t - T0) * 12, (t - T0) * 5)\n            g = np.broadcast_to(F[:, None, None, None], (8, S, nD, L)) + rng.normal(0, 0.15, (8, S, nD, L))\n            bm = np.full((8, S, nD, L), mass, np.float32) * (1 + rng.normal(0, 0.05, (8, S, nD, L)))\n            np.savez(f\'{dirn}/{m}@{a}.npz\', g_ell=g.astype(np.float32), g_ban_mass=bm.astype(np.float32),\n                     g_banned_top1=np.zeros((8, S, nD, L), bool), toks=rng.integers(0, 1000, (8, S, nD, L)).astype(np.int32),\n                     prompts=rng.integers(0, 1000, (nD, 8)).astype(np.int32), temps=np.array(B7A_TEMPS), use=np.arange(nD))\n\n    with tempfile.TemporaryDirectory() as dirn:\n        write(dirn, \'pythia-1b\', 5.0, 1.5)\n        write(dirn, \'olmo2-1b\', 5.0, 5.0)\n        write(dirn, \'qwen2.5-1.5b\', 5.0, 0.0)\n        r = analyze7b(dirn)\n    w = {m: r[\'models\'][m][\'BQ1\'][\'word\'] for m in B7A_MODELS}\n    assert w[\'pythia-1b\'] == \'写しの禁止のほうがよく効く\', r[\'models\'][\'pythia-1b\'][\'BQ1\']\n    assert w[\'olmo2-1b\'] == \'同じくらい（束一的）\', r[\'models\'][\'olmo2-1b\'][\'BQ1\']\n    assert w[\'qwen2.5-1.5b\'] == \'写しの禁止のほうがよく効く\', r[\'models\'][\'qwen2.5-1.5b\'][\'BQ1\']\n    assert r[\'BQ1\'] == 2\n    assert r[\'models\'][\'olmo2-1b\'][\'BQ2_word\'] in DOWN_WORDS and r[\'models\'][\'qwen2.5-1.5b\'][\'BQ2_word\'] not in DOWN_WORDS\n    assert log_q(1.0, -0.1) == float(\'inf\') and np.isnan(log_q(-1.0, -1.0))\n    return \'OK\'\n'
assert hashlib.sha256(SRC7BA.encode()).hexdigest() == 'd4da9ac740e2b6451845da8496ff90e451d5ba11bbcef28e1206da2cc61d1135'
exec(SRCA22)
exec(SRCA24)
exec(SRC7BA)
print('自己テスト：', self_test_a27b_analyze())


In [ ]:
# セル3：分析して保存（結果は表示しない）
import json
try:
    r = analyze7b('/content/drive/MyDrive/logit-thermo/A2_7B')
    r['a2_7b_analyze_sha256'] = 'd4da9ac740e2b6451845da8496ff90e451d5ba11bbcef28e1206da2cc61d1135'
    json.dump(_clean(r), open('/content/drive/MyDrive/logit-thermo/A2_7B' + '/a2_7b_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-7b の分析で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：判定と要約を A2_7B/a2_7b_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
